# Program for running back-classification of NACE on test data
1. Read in data
2. Classify 1-to-1
3. Identify problem manual groups
4. Classify using rules
5. Classify using machine learning model

To do:
- add in flag variable
- Move things to config

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
import os
import s3fs

import mlflow

import nltk
from nltk.corpus import stopwords

from config import bucket_info

In [2]:
from utils.one_to_one import get_1to1_dict, get_problem_NACE
from utils.rules_classification import norwegian_rules, classify_rules
from utils.ml_classification import get_label, get_possibles
from utils.text_processing import preprocess

In [3]:
# Location for model and data
bucket_url = bucket_info["bucket_url"]
bucket = "projet-aiml4os-wp10/Cluster5"

test_path = f"{bucket_url}/test_doublenace_2026-03-23.parquet"
conversion_path = f"{bucket_url}/NACE2.1-NACE2_Table_V1.05.xlsx"

In [4]:
# Set model id and paths
# Model URI: models:/m-6dd8a7b6c4454de19e8eb65688f6e2b2

model_id = "m-6dd8a7b6c4454de19e8eb65688f6e2b2"
model_path = f"s3://{bucket}/mlflow_artifacts/models/{model_id}/artifacts"
os.environ["MLFLOW_S3_ENDPOINT_URL"] = os.environ["S3_ENDPOINT"]

In [5]:
os.environ["S3_ENDPOINT"] = "https://minio.lab.sspcloud.fr"
os.environ["AWS_S3_ENDPOINT"] = "https://minio.lab.sspcloud.fr"
os.environ["MLFLOW_S3_ENDPOINT_URL"] = os.environ["S3_ENDPOINT"]
os.environ["AWS_DEFAULT_REGION"] = 'us-east-1'

In [6]:
os.getenv('S3_ENDPOINT')

'https://minio.lab.sspcloud.fr'

In [7]:
from getpass import getpass

os.environ["AWS_ACCESS_KEY_ID"] = getpass("AWS_ACCESS_KEY_ID: ")
os.environ["AWS_SECRET_ACCESS_KEY"] = getpass("AWS_SECRET_ACCESS_KEY: ")
os.environ["AWS_SESSION_TOKEN"] = getpass("AWS_SESSION_TOKEN: ")


AWS_ACCESS_KEY_ID:  ········
AWS_SECRET_ACCESS_KEY:  ········
AWS_SESSION_TOKEN:  ········


In [8]:
import os

for key in [
    "AWS_ACCESS_KEY_ID",
    "AWS_SECRET_ACCESS_KEY",
    "S3_ENDPOINT",
    "MLFLOW_S3_ENDPOINT_URL",
]:
    print(f"{key}: {'SET' if os.environ.get(key) else 'MISSING'}")

AWS_ACCESS_KEY_ID: SET
AWS_SECRET_ACCESS_KEY: SET
S3_ENDPOINT: SET
MLFLOW_S3_ENDPOINT_URL: SET


In [9]:
# Set seed
seed = 42

### Read in data

In [10]:
test = pd.read_parquet(test_path)
print(f'Number of observations in test: {test.shape[0]}')

Number of observations in test: 110182


In [11]:
# Drop NACE 2 code for demonstartion purposes on how this can be predicted
test = test.drop(columns="nace_20_code")

### Classify 1-to-1

In [12]:
# Create dictionary of additional 1-to-1 relationships which are country/regional level
country_dict = {
        "00.00": "00.00",
        "01.41": "01.41",
        "01.63": "01.63",
        "10.62": "10.62",
        "11.07": "11.07",
        "25.21": "25.21",
        "26.70": "26.70",
        "26.60": "32.50",
        "55.90": "55.90",
        "58.19": "58.19",
        "77.22": "77.29",
        "78.20": "78.20",
    }

# Get standard dictionary of 1-to-1 relationships
one_to_one_dict = get_1to1_dict(conversion_path, country_dict) 

In [13]:
# Apply to data
test["nace_20_code"] = test.nace_21_code.map(one_to_one_dict)
test["nace_20_source"] = np.where(test.nace_20_code.notna(), "one-to-one", pd.NA)

n_1to1 = test.nace_20_code.notna().sum()
print(f'Number of observations classifed using 1-to-1: {n_1to1}')

Number of observations classifed using 1-to-1: 86887


### Identify problem NACE for manual coding

In [14]:
problem_nace = get_problem_NACE(conversion_path)

In [15]:
mask_problem = test.nace_21_code.isin(problem_nace)
test.loc[mask_problem, "nace_20_code"] = "manual"
test.loc[mask_problem, "nace_20_source"] = "manual"

In [16]:
print(f'Number of observations in problem groups: {mask_problem.sum()}')

Number of observations in problem groups: 0


### Classify using rules


In [17]:
# Create text variable
test["company_name"] = test["company_name"].fillna("")
test["company_activity"] = test["company_activity"].fillna("")
test["company_name_description"] = test["company_name"] + test["company_activity"]

In [18]:
# Classify
nace_20_rules = classify_rules(test, keyword_rules=norwegian_rules, company_nace21="nace_21_code", 
        company_text="company_name_description")
test["nace_20_code"] = test["nace_20_code"].fillna(nace_20_rules)
test["nace_20_source"] = np.where(nace_20_rules.notna(), "rules", test.nace_20_source)

In [19]:
print(f'Number of observations with rules classification: {nace_20_rules.notna().sum()}')

Number of observations with rules classification: 7


### Classify using machine learning model


In [20]:
print(f'Number of observations requiring classification with ML model: {test.nace_20_code.isna().sum()}')

Number of observations requiring classification with ML model: 23288


**Preprocess test**

In [21]:
# Get stop words
nltk.download('stopwords') # one time to use locally
norwegian_stopwords = set(stopwords.words('norwegian'))|{"as"}

# Create text variable
test["company_name"] = test["company_name"].fillna("")
test["company_activity"] = test["company_activity"].fillna("")
test["company_text"] = test["company_name"] + " " + test["company_activity"]

# Preprocess and add in nace 2.1
#text_variable, stopwords, nace_21_variable = "nace_21_code"
test["company_text_processed"] = preprocess(test, 
                                             text_variable="company_text", 
                                             stopwords = norwegian_stopwords, 
                                             nace_21_variable = "nace_21_code"
                                            )

[nltk_data] Downloading package stopwords to /home/onyxia/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


**Fetch model**

In [22]:

for key in [
    "AWS_ACCESS_KEY_ID",
    "AWS_SECRET_ACCESS_KEY",
    "S3_ENDPOINT",
    "MLFLOW_S3_ENDPOINT_URL",
]:
    print(f"{key}: {'SET' if os.environ.get(key) else 'MISSING'}")

AWS_ACCESS_KEY_ID: SET
AWS_SECRET_ACCESS_KEY: SET
S3_ENDPOINT: SET
MLFLOW_S3_ENDPOINT_URL: SET


In [23]:
model = mlflow.sklearn.load_model(model_path)

In [24]:
# Predict using model (prediction matrix)
pred_mat = model.predict_proba(test.company_text_processed)

In [25]:
pred_mat

array([[0.38850245, 0.36743454, 0.04415288, 0.19991013],
       [0.09427572, 0.86637776, 0.0251942 , 0.01415232],
       [0.06019077, 0.10257709, 0.09493064, 0.7423015 ],
       ...,
       [0.31905597, 0.53547118, 0.03780056, 0.10767229],
       [0.20401496, 0.49259878, 0.04254012, 0.26084613],
       [0.2807703 , 0.59512801, 0.08288937, 0.04121232]],
      shape=(110182, 4))

In [26]:
# Select from only the possible NACE 2.0 codes, based on NACE rev. 2.1
all_labels = model.classes_
possibles = get_possibles(test, company_nace21 = "nace_21_code", conversion_path=conversion_path)
labels_mat = [[l for l in all_labels] for _ in range(test.shape[0])]

# Predict NACE: simplify this function - dont need labels_svc
ml_labels = get_label(labels_mat, pred_mat, possibles, all_labels, choose_type = "random", seed = seed)



In [32]:
type(ml_labels)

tuple

In [44]:
#pd.Series(ml_labels[0]).isna().sum()

np.int64(109613)

In [45]:
#len(ml_labels[0])

110182

In [58]:
len(test)

110182

In [29]:
#test["nace_20_source"].value_counts()

nace_20_source
one-to-one    86887
rules             7
Name: count, dtype: int64

In [46]:

test["nace_20_source"] = np.where(test.nace_20_code.isna(), "ml", test.nace_20_source)

In [47]:
test["nace_20_code"] = test["nace_20_code"].fillna(pd.Series(ml_labels[0]))

In [48]:
print(f"Number still missing nace 2.0: {test.nace_20_code.isna().sum()}")

Number still missing nace 2.0: 22756


In [57]:
test

,orgnr,company_name,company_activity,company_purpose,language,number_of_employees,orgform,orgform_name_nb,orgform_name_en,date_of_incorporation,...,sector_name_nb,sector_name_en,nace_21_code,nace_21_name_nb,nace_21_name_en,nace_20_code,nace_20_source,company_name_description,company_text,company_text_processed
0,989209981,ALOPO AS,Investeringer i andre virksomheter i form av a...,Investeringer herunder deltagelse i andre virk...,Bokmål,<NA>,AS,Aksjeselskap,Limited company,2005-12-20,...,Øvrige finansielle foretak unntatt forsikring,"Other financial enterprises, except insurance ...",64.32,Andre egeninvesteringsselskaper,"Activities of trust, estate and agency accounts",64.30,one-to-one,ALOPO ASInvesteringer i andre virksomheter i f...,ALOPO AS Investeringer i andre virksomheter i ...,alo lop opo po_ o_i _in inv nve ves est ste te...
1,920253261,KAREN MARGRETHE KEMI NYHEIM,"Guiding, doudji (samisk håndverk) med mer.",None,Bokmål,<NA>,ENK,Enkeltpersonforetak,Sole proprietorship,None,...,Personlig næringsdrivende,Unincorporated enterprises within households,79.90,Guide- og reiseledervirksomhet,Other reservation service and related activities,79.90,one-to-one,"KAREN MARGRETHE KEMI NYHEIMGuiding, doudji (sa...","KAREN MARGRETHE KEMI NYHEIM Guiding, doudji (s...",kar are ren en_ n_m _ma mar arg rgr gre ret et...
2,922675341,KEBELLI AS,Investeringer i andre selskaper og annen virks...,"Eie aksjer i ett eller flere selskaper, og for...",Bokmål,<NA>,AS,Aksjeselskap,Limited company,2019-02-01,...,Private aksjeselskaper mv.,Private non-financial incorporated enterprises,00.00,Uoppgitt,None,00.00,one-to-one,KEBELLI ASInvesteringer i andre selskaper og a...,KEBELLI AS Investeringer i andre selskaper og ...,keb ebe bel ell lli li_ i_i _in inv nve ves es...
3,985049173,NORWEGIAN SERVICE TEAM CONTRACTING AS,"Alt innen overflatebehandling, høytrykksrengjø...","Alt innen overflatebehandling, høytrykksrengjø...",Bokmål,<NA>,AS,Aksjeselskap,Limited company,2002-10-24,...,Private aksjeselskaper mv.,Private non-financial incorporated enterprises,25.51,Overflatebehandling av metaller,Coating of metals,25.61,one-to-one,NORWEGIAN SERVICE TEAM CONTRACTING ASAlt innen...,NORWEGIAN SERVICE TEAM CONTRACTING AS Alt inne...,nor orw rwe weg egi gia ian an_ n_s _se ser er...
4,929417852,TURMAT OG VILLEVEKSTER AS,Kurs og foredrag om: tørking av turmat/ genere...,"Utdanning og foredrag, salg av ferskvare, tørk...",Bokmål,<NA>,AS,Aksjeselskap,Limited company,2022-06-03,...,Private aksjeselskaper mv.,Private non-financial incorporated enterprises,85.59,Annen undervisning ellers,Other education n.e.c.,85.59,one-to-one,TURMAT OG VILLEVEKSTER ASKurs og foredrag om: ...,TURMAT OG VILLEVEKSTER AS Kurs og foredrag om:...,tur urm rma mat at_ t_v _vi vil ill lle lev ev...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
110177,927465361,BILVASK.NO REVETAL AS,Bilvask.,"Drift av bilvaskeanlegg, deltakelse i andre se...",Bokmål,<NA>,AS,Aksjeselskap,Limited company,2021-06-24,...,Private aksjeselskaper mv.,Private non-financial incorporated enterprises,95.31,Reparasjon og vedlikehold av motorvogner,Repair and maintenance of motor vehicles,45.20,one-to-one,BILVASK.NO REVETAL ASBilvask.,BILVASK.NO REVETAL AS Bilvask.,bil ilv lva vas ask skn kno no_ o_r _re rev ev...
110178,989231251,BREKKO EIENDOM AS,Selskapets formål er eiendomsutvikling og eien...,Selskapets formål er eiendomsutvikling og eien...,Bokmål,<NA>,AS,Aksjeselskap,Limited company,2005-12-23,...,Private aksjeselskaper mv.,Private non-financial incorporated enterprises,79.12,Reisearrangørvirksomhet ellers,Tour operator activities,79.12,one-to-one,BREKKO EIENDOM ASSelskapets formål er eiendoms...,BREKKO EIENDOM AS Selskapets formål er eiendom...,bre rek ekk kko ko_ o_e _ei eie ien end ndo do...
110179,989060090,BOLSILLO AS,Investering og deltagelse i andre virksomheter...,Investering og deltagelse i andre virksomheter.,Bokmål,<NA>,AS,Aksjeselskap,Limited company,2005-11-10,...,Øv

In [63]:
test[(test['nace_20_code'].notna()) & (test['nace_20_source']=='ml')]

,orgnr,company_name,company_activity,company_purpose,language,number_of_employees,orgform,orgform_name_nb,orgform_name_en,date_of_incorporation,...,sector_name_nb,sector_name_en,nace_21_code,nace_21_name_nb,nace_21_name_en,nace_20_code,nace_20_source,company_name_description,company_text,company_text_processed
210,916398360,LEIRVIKFLATEN 19 AS,"Eie, drive, og investere i fast eiendom, herun...","Eie, drive, og investere i fast eiendom, herun...",Bokmål,<NA>,AS,Aksjeselskap,Limited company,1978-01-03,...,Private aksjeselskaper mv.,Private non-financial incorporated enterprises,25.53,Bearbeiding av metaller,Machining of metals,25.61,ml,"LEIRVIKFLATEN 19 ASEie, drive, og investere i ...","LEIRVIKFLATEN 19 AS Eie, drive, og investere i...",lei eir irv rvi vik ikf kfl fla lat ate ten en...
242,955421086,HØYANGER MØBELTAPETSERING Anne Gunn Fossheim,"Rep. av møblar, salg av møbelstof, gardiner m.m.",None,Nynorsk,<NA>,ENK,Enkeltpersonforetak,Sole proprietorship,None,...,Personlig næringsdrivende,Unincorporated enterprises within households,31.00,Produksjon av møbler,Manufacture of furniture,31.02,ml,HØYANGER MØBELTAPETSERING Anne Gunn FossheimRe...,HØYANGER MØBELTAPETSERING Anne Gunn Fossheim R...,høy øya yan ang nge ger er_ r_m _mø møb øbe be...
512,915021921,SMELAND DESIGN AS,Utvikle og produsere møbel/snekkerprodukter og...,Utvikle og produsere møbel/snekkerprodukter og...,Bokmål,<NA>,AS,Aksjeselskap,Limited company,2015-02-11,...,Private aksjeselskaper mv.,Private non-financial incorporated enterprises,31.00,Produksjon av møbler,Manufacture of furniture,31.02,ml,SMELAND DESIGN ASUtvikle og produsere møbel/sn...,SMELAND DESIGN AS Utvikle og produsere møbel/s...,sme mel ela lan and nd_ d_d _de des esi sig ig...
566,917937699,LANGÅKER MASKINERINGSTEKNIKK,"CNC maskinering av deler i aluminium, plast og...",None,Bokmål,<NA>,ENK,Enkeltpersonforetak,Sole proprietorship,None,...,Personlig næringsdrivende,Unincorporated enterprises within households,25.53,Bearbeiding av metaller,Machining of metals,25.61,ml,LANGÅKER MASKINERINGSTEKNIKKCNC maskinering av...,LANGÅKER MASKINERINGSTEKNIKK CNC maskinering a...,lan ang ngå gåk åke ker er_ r_m _ma mas ask sk...
804,911004836,CASPERSENS TRYKKERI AS,Trykkerivirksomhet og hva som dermed står i fo...,Trykkerivirksomhet og hva som dermed står i fo...,Bokmål,<NA>,AS,Aksjeselskap,Limited company,1969-01-01,...,Private aksjeselskaper mv.,Private non-financial incorporated enterprises,18.12,Trykking ellers,Other printing,18.12,ml,CASPERSENS TRYKKERI ASTrykkerivirksomhet og hv...,CASPERSENS TRYKKERI AS Trykkerivirksomhet og h...,cas asp spe per ers rse sen ens ns_ s_t _tr tr...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
108126,993129607,ASBJØRN KROGSTAD HOLDING AS,Investeringer i aksjer og andre finansielle in...,"Deltagelse i aksjeselskaper, erverv av aksjepo...",Bokmål,<NA>,AS,Aksjeselskap,Limited company,2008-09-17,...,Private aksjeselskaper mv.,Private non-financial incorporated enterprises,25.53,Bearbeiding av metaller,Machining of metals,25.61,ml,ASBJØRN KROGSTAD HOLDING ASInvesteringer i aks...,ASBJØRN KROGSTAD HOLDING AS Investeringer i ak...,asb sbj bjø jør ørn rn_ n_k _kr kro rog ogs gs...
108877,925469831,EMET AS,"Drive produksjon, handel samt hva dermed står ...","Drive produksjon, handel samt hva dermed står ...",Bokmål,10,AS,Aksjeselskap,Limited company,1975-01-03,...,Private aksjeselskaper mv.,Private non-financial incorporated enterprises,25.53,Bearbeiding av metaller,Machining of metals,25.61,ml,"EMET ASDrive produksjon, handel samt hva derme...","EMET AS Drive produksjon, handel samt hva derm...",eme met et_ t_d _dr dri riv ive ve_ e_p _pr pr...
109205,916818580,FLEXIPRINT AS,Trykking ellers.,Produsere og selge skilt og dekor og yte tilkn...,Bokmål,11,AS,Aksjeselskap,Limited company,2016-02-10,...,Private aksjeselskaper mv.,Private non-financial incorporated enterprises,18.12,Trykking ellers,Other printing,18.12,ml,FLEXIPRINT ASTrykking ellers.